# Explicit arity non binary bottom up oracle with Stanza trees

This notebook gives a compact implementation of the previous non binary bottom up baseline in the style of González Rodríguez et al. 2019.  The system does not use delimiters.  Instead, each reduction action encodes both the nonterminal label and the arity of the constituent being built.

The pipeline is parallel to the binary bottom up and delimiter guided bottom up notebooks:

1. read a PTB style tree with Stanza,
2. extract a static bottom up oracle,
3. apply the oracle to reconstruct a Stanza tree,
4. check exact tree reconstruction.

All example trees in this notebook are synthetic.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Iterable, List, Optional, Sequence, Tuple

from stanza.models.constituency.parse_tree import Tree
from stanza.models.constituency.tree_reader import read_trees


# ============================================================
# Stanza tree utilities
# ============================================================

def bracket_diagnostics(ptb_string: str) -> dict:
    """Return a small diagnostic report for PTB bracket balance.

    This is only a precheck for examples pasted into the notebook.
    Stanza's ``read_trees`` remains the actual tree reader.
    """
    balance = 0
    first_extra_close = None

    for line_no, line in enumerate(ptb_string.splitlines(), start=1):
        for col_no, char in enumerate(line, start=1):
            if char == "(":
                balance += 1
            elif char == ")":
                balance -= 1
                if balance < 0 and first_extra_close is None:
                    first_extra_close = (line_no, col_no)

    return {
        "open_parens": ptb_string.count("("),
        "close_parens": ptb_string.count(")"),
        "final_balance": balance,
        "first_extra_close": first_extra_close,
    }


def assert_balanced_ptb(ptb_string: str) -> None:
    """Raise a clear error if a pasted PTB string has unmatched parentheses."""
    report = bracket_diagnostics(ptb_string)
    if report["first_extra_close"] is not None:
        line_no, col_no = report["first_extra_close"]
        raise ValueError(
            "Malformed PTB string: extra closing parenthesis at "
            f"line {line_no}, column {col_no}. "
            f"opens={report['open_parens']}, closes={report['close_parens']}."
        )
    if report["final_balance"] != 0:
        raise ValueError(
            "Malformed PTB string: unbalanced parentheses. "
            f"opens={report['open_parens']}, closes={report['close_parens']}, "
            f"final_balance={report['final_balance']}."
        )


def parse_ptb(ptb_string: str) -> Tree:
    """Read exactly one PTB style bracketed tree with Stanza."""
    ptb_string = ptb_string.strip()
    assert_balanced_ptb(ptb_string)
    trees = list(read_trees(ptb_string, use_tqdm=False))
    if len(trees) != 1:
        raise ValueError(f"Expected exactly one tree, found {len(trees)}.")
    return trees[0]


def make_tree(label: str, children: Optional[Sequence[Tree]] = None) -> Tree:
    """
    Construct a Stanza constituency tree.

    Stanza's constituency Tree is used both for internal nodes and leaves:
    a terminal word is represented as Tree(word, []).
    """
    return Tree(label, [] if children is None else list(children))


def label(node: Tree) -> str:
    value = node.label
    return value() if callable(value) else value


def children(node: Tree) -> List[Tree]:
    value = node.children
    return list(value() if callable(value) else value)


def is_leaf(node: Tree) -> bool:
    return len(children(node)) == 0


def is_preterminal(node: Tree) -> bool:
    ch = children(node)
    return len(ch) == 1 and is_leaf(ch[0])


def terminal_of(preterminal: Tree) -> str:
    if not is_preterminal(preterminal):
        raise ValueError(f"Expected preterminal, found {to_ptb(preterminal)}")
    return label(children(preterminal)[0])


def yield_words(tree: Tree) -> List[str]:
    if is_leaf(tree):
        return [label(tree)]

    words: List[str] = []
    for child in children(tree):
        words.extend(yield_words(child))
    return words


def to_ptb(tree: Tree) -> str:
    """Return a compact PTB bracket string."""
    ch = children(tree)
    if not ch:
        return label(tree)
    return f"({label(tree)} {' '.join(to_ptb(child) for child in ch)})"


def pretty(tree: Tree) -> str:
    """Use Stanza's pretty printer when available, otherwise compact PTB."""
    if hasattr(tree, "pformat"):
        return tree.pformat()
    return to_ptb(tree)


def normalize_bracket_token(token: str) -> str:
    """Normalize literal bracket tokens to PTB bracket symbols."""
    return {
        "(": "-LRB-",
        ")": "-RRB-",
        "{": "-LCB-",
        "}": "-RCB-",
        "[": "-LSB-",
        "]": "-RSB-",
    }.get(token, token)


def assert_same_tree(left: Tree, right: Tree) -> None:
    left_ptb = to_ptb(left)
    right_ptb = to_ptb(right)
    if left_ptb != right_ptb:
        raise AssertionError(
            "Trees differ.\n\n"
            f"Gold:\n{left_ptb}\n\n"
            f"Pred:\n{right_ptb}"
        )

## Transition representation

The explicit arity baseline uses no delimiter and no separate nonterminal marker.  A reduction action is written as `REDUCE-X#k`, where `X` is the phrase or preterminal label and `k` is the number of completed stack items to reduce.

In [ ]:
OracleOp = Tuple[str, Optional[str], Optional[int]]

SHIFT = "SHIFT"
REDUCE = "REDUCE"
FINISH = "FINISH"


def format_action(action: OracleOp) -> str:
    op, value, arity = action
    if op == SHIFT:
        return f"SHIFT({value})"
    if op == REDUCE:
        return f"REDUCE-{value}#{arity}"
    if op == FINISH:
        return FINISH
    return f"{op}({value}, {arity})"


def print_oracle(oracle: Sequence[OracleOp]) -> None:
    for i, action in enumerate(oracle, 1):
        print(f"{i:3d}. {format_action(action)}")


def action_counts(oracle: Sequence[OracleOp]) -> dict:
    counts = {SHIFT: 0, REDUCE: 0, FINISH: 0}
    for op, _value, _arity in oracle:
        counts[op] = counts.get(op, 0) + 1
    return counts


def reduce_inventory(oracle: Sequence[OracleOp]) -> List[str]:
    """Return the distinct explicit arity reduce actions observed in an oracle."""
    inventory = {
        format_action(action)
        for action in oracle
        if action[0] == REDUCE
    }
    return sorted(inventory)

## Static oracle

The oracle traverses the gold tree bottom up.  For a preterminal `X(w)`, it emits `SHIFT(w)` followed by `REDUCE-X#1`.  For a phrase `X(C1, ..., Ck)`, it constructs all children from left to right and then emits `REDUCE-X#k`.  Thus the arity is part of the action itself.

In [ ]:
def get_oracle_previous_nonbinary_bottomup(tree: Tree) -> List[OracleOp]:
    """
    Produce a static oracle for the explicit arity non binary bottom up baseline.

    The transition inventory is:

        SHIFT(w)
        REDUCE-X#k
        FINISH

    There is no delimiter and no separate NT action.
    """
    oracle: List[OracleOp] = []

    def visit(node: Tree) -> None:
        ch = children(node)

        if is_leaf(node):
            oracle.append((SHIFT, label(node), None))
            return

        node_label = label(node)

        if is_preterminal(node):
            oracle.append((SHIFT, terminal_of(node), None))
            oracle.append((REDUCE, node_label, 1))
            return

        for child in ch:
            visit(child)

        oracle.append((REDUCE, node_label, len(ch)))

    visit(tree)
    oracle.append((FINISH, None, None))
    return oracle

## Applying the oracle

The interpreter reconstructs a Stanza tree from the sentence and oracle.  Stack items are completed Stanza `Tree` objects.  A `REDUCE-X#k` action pops the top `k` completed trees and pushes `X(...)`, preserving left to right child order.

In [ ]:
def apply_oracle_previous_nonbinary_bottomup(
    sentence: Sequence[str],
    oracle: Sequence[OracleOp],
    verbose: bool = False,
) -> Tree:
    stack: List[Tree] = []
    word_idx = 0
    finished = False

    def stack_string() -> str:
        if not stack:
            return "[]"
        return "[ " + " | ".join(to_ptb(item) for item in stack) + " ]"

    def buffer_string() -> str:
        if word_idx >= len(sentence):
            return "[]"
        return "[ " + " ".join(sentence[word_idx:]) + " ]"

    def trace(step: int, action: OracleOp, when: str) -> None:
        if not verbose:
            return
        print(f"\nStep {step}: {format_action(action)} [{when}]")
        print(f"Buffer: {buffer_string()}")
        print(f"Stack : {stack_string()}")

    def do_shift(value: Optional[str]) -> None:
        nonlocal word_idx

        if word_idx >= len(sentence):
            raise ValueError("SHIFT exceeds sentence length.")

        expected = normalize_bracket_token(sentence[word_idx])
        observed = normalize_bracket_token(value if value is not None else sentence[word_idx])

        if observed != expected:
            raise ValueError(
                f"SHIFT mismatch at position {word_idx}: "
                f"oracle has {observed!r}, sentence has {expected!r}."
            )

        stack.append(make_tree(expected))
        word_idx += 1

    def do_reduce(value: Optional[str], arity: Optional[int]) -> None:
        if value is None:
            raise ValueError("REDUCE requires a label.")
        if arity is None:
            raise ValueError("REDUCE requires an explicit arity.")
        if arity <= 0:
            raise ValueError(f"REDUCE arity must be positive, found {arity}.")
        if len(stack) < arity:
            raise ValueError(
                f"REDUCE-{value}#{arity} requires {arity} stack items, "
                f"but stack has only {len(stack)}."
            )

        reduced_children = stack[-arity:]
        del stack[-arity:]

        # For bracket POS tags, keep the terminal in PTB-normalized form.
        if arity == 1 and value in {"-LRB-", "-RRB-", "-LCB-", "-RCB-", "-LSB-", "-RSB-"}:
            child = reduced_children[0]
            if is_leaf(child):
                reduced_children = [make_tree(value)]

        stack.append(make_tree(value, reduced_children))

    if verbose:
        print("=== APPLY EXPLICIT ARITY NON BINARY BOTTOM-UP ORACLE ===")
        print("Sentence:", list(sentence))
        print("Oracle:")
        print_oracle(oracle)

    for step, action in enumerate(oracle, 1):
        op, value, arity = action

        if finished:
            raise ValueError("No action may follow FINISH.")

        trace(step, action, "before")

        if op == SHIFT:
            do_shift(value)
        elif op == REDUCE:
            do_reduce(value, arity)
        elif op == FINISH:
            if word_idx != len(sentence):
                raise ValueError(
                    f"Cannot FINISH before consuming all words: "
                    f"{word_idx} of {len(sentence)} consumed."
                )
            if len(stack) != 1:
                raise ValueError(f"Cannot FINISH with stack {stack_string()}.")
            finished = True
        else:
            raise ValueError(f"Unknown action: {op!r}")

        trace(step, action, "after")

    if not finished:
        raise ValueError("Oracle ended without FINISH.")

    return stack[0]

## End to end helper

In [ ]:
@dataclass
class PreviousNonbinaryBottomUpResult:
    gold_tree: Tree
    sentence: List[str]
    oracle: List[OracleOp]
    reconstructed_tree: Tree

    @property
    def gold_ptb(self) -> str:
        return to_ptb(self.gold_tree)

    @property
    def reconstructed_ptb(self) -> str:
        return to_ptb(self.reconstructed_tree)


def run_previous_nonbinary_bottomup_pipeline(
    ptb_string: str,
    verbose: bool = False,
    check: bool = True,
) -> PreviousNonbinaryBottomUpResult:
    gold_tree = parse_ptb(ptb_string)
    sentence = yield_words(gold_tree)
    oracle = get_oracle_previous_nonbinary_bottomup(gold_tree)
    reconstructed_tree = apply_oracle_previous_nonbinary_bottomup(
        sentence=sentence,
        oracle=oracle,
        verbose=verbose,
    )

    result = PreviousNonbinaryBottomUpResult(
        gold_tree=gold_tree,
        sentence=sentence,
        oracle=oracle,
        reconstructed_tree=reconstructed_tree,
    )

    if check:
        assert_same_tree(gold_tree, reconstructed_tree)

    return result

## Minimal example

In [ ]:
ptb_string = '(TOP (S (NP (NNP Mira) (NNP Vale)) (VP (VBZ builds) (NP (NN robots))) (. .)))'

result = run_previous_nonbinary_bottomup_pipeline(ptb_string, verbose=False)

print("Sentence:")
print(result.sentence)

print("\nOracle:")
print_oracle(result.oracle)

print("\nAction counts:")
print(action_counts(result.oracle))

print("\nReconstructed tree:")
print(result.reconstructed_ptb)

print("\nSame tree?")
print(result.gold_ptb == result.reconstructed_ptb)

## Larger example

In [ ]:
ptb_string = '(TOP (S (NP (DT The) (JJ young) (NN fox)) (VP (VBD watched) (SBAR (IN as) (S (NP (DT a) (NN bird)) (VP (VBD flew) (PP (IN over) (NP (DT the) (NN pond))))))) (. .)))'

result = run_previous_nonbinary_bottomup_pipeline(ptb_string, verbose=False)

print("Sentence length:", len(result.sentence))
print("Number of oracle actions:", len(result.oracle))
print("Action counts:", action_counts(result.oracle))
print("Distinct reduce actions:", len(reduce_inventory(result.oracle)))
print("Same tree?", result.gold_ptb == result.reconstructed_ptb)
print("\nReconstructed tree:")
print(result.reconstructed_ptb)

## Regression checks

In [ ]:
examples = [
    '(TOP (S (NP (NNP Mira) (NNP Vale)) (VP (VBZ builds) (NP (NN robots))) (. .)))',
    '(TOP (S (NP (DT The) (JJ young) (NN fox)) (VP (VBD watched) (SBAR (IN as) (S (NP (DT a) (NN bird)) (VP (VBD flew) (PP (IN over) (NP (DT the) (NN pond))))))) (. .)))',
    '(TOP (S (VP (VB Wait))))',
    '(TOP (S (NP (NNP Ivo)) (VP (VBZ packs) (NP (NNS maps)) (NP (NNS snacks)) (PP (IN for) (NP (NN travel)))) (. .)))',
]

for i, example in enumerate(examples, 1):
    result = run_previous_nonbinary_bottomup_pipeline(example)
    assert result.gold_ptb == result.reconstructed_ptb
    print(f"Example {i}: OK, {len(result.oracle)} actions, {len(reduce_inventory(result.oracle))} distinct reduce actions")

## Paper note

The explicit arity baseline learns a distinct reduction action for each observed pair of nonterminal label and arity.  In contrast, the delimiter guided system separates label prediction from span selection: the nonterminal is introduced independently and the reduction span is recovered structurally from the delimiter.  This notebook therefore serves as the non binary baseline without `D`.